In [ ]:
from pathlib import Path
import pandas as pd 
import glob
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()
import xgboost as xgb
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from scipy.stats import pearsonr, spearmanr

root = Path('/data/data/malpolon/xgb')
inputs_path = Path('/marbec-data/RLS-Australia/malpolon/inputs/australia/')
output_path = Path('/marbec-data/RLS-Australia/malpolon/outputs/')

In [13]:
fulldf = pd.read_csv(inputs_path / 'database_common.csv', index_col='survey_id',
                     dtype = {22:str, 24:str, 25:str})
species = list(fulldf.columns[-818:-1])
groundtruth = fulldf[species]

In [20]:
## Bins data
groundtruth_bins = {}
for i in [5, 10, 20]:
    df = pd.read_csv(inputs_path / f"database_common_{i}binned.csv", index_col='survey_id')
    groundtruth_bins[i] = df[species]

/tmp/ipykernel_93855/1370679637.py:4: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(inputs_path / f"database_common_{i}binned.csv", index_col='survey_id')
/tmp/ipykernel_93855/1370679637.py:4: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(inputs_path / f"database_common_{i}binned.csv", index_col='survey_id')
/tmp/ipykernel_93855/1370679637.py:4: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(inputs_path / f"database_common_{i}binned.csv", index_col='survey_id')


## Preparation

In [14]:
## Calculate predictors
 
def get_compound_values(row):

    survey_id = row.name

    filename = Path(inputs_path) / "env" / (str(survey_id) + '.npy')
    x = np.load(filename).astype(np.float32)
    env = np.transpose(x, (2, 0, 1))

    filename = Path(inputs_path) / "hum" / (str(survey_id) + '.npy')
    x = np.load(filename).astype(np.float32)
    hum = np.transpose(x, (2, 0, 1))

    envhum = np.concatenate([env, hum], axis=0)

    center_values = 0.25 * (envhum[:, 16, 16] + envhum[:, 15, 16] + envhum[:, 16, 15] + envhum[:, 15, 15])
    means = np.mean(envhum, axis=(1,2))
    deviations = np.std(envhum, axis=(1,2)) / means


    filename = Path(inputs_path) / "dhw" / (str(survey_id) + '.npy')
    dhw = np.load(filename).astype(np.float32)
    dhw_vect = np.array([np.mean(dhw), np.std(dhw) / np.mean(dhw), np.max(dhw)])

    return np.concatenate([center_values, means, deviations, dhw_vect])

# X = fulldf[['subset']].progress_apply(get_compound_values, axis=1, result_type='expand')
# X.to_csv(root / f'X_compound_values_mm.csv')

In [22]:
## Prepare datasets

def get_datasets(sp, modeltype='rf', objective='pa', num_bins=None):

    # X
    X = pd.read_csv(root / f'X_compound_values_mm.csv', index_col = 0)
    X_train = X.loc[fulldf['subset'] == 'train']
    X_val = X.loc[fulldf['subset'] == 'val']
    X_test = X.loc[fulldf['subset'] == 'test']

    # Y
    if objective == 'pa':
        targets = (groundtruth[sp] > 0).astype("category")
    elif objective == 'bins':
        targets = groundtruth_bins[num_bins][sp].astype("category")
    else:
        targets = groundtruth[sp].astype(float)

    Y_train = targets.loc[fulldf['subset'] == 'train']
    Y_val = targets.loc[fulldf['subset'] == 'val']
    Y_test = targets.loc[fulldf['subset'] == 'test']

    if modeltype == 'xgb':

        dtrain_clf = xgb.DMatrix(X_train, Y_train, enable_categorical=True)
        dval_clf = xgb.DMatrix(X_val, Y_val, enable_categorical=True)
        dtest_clf = xgb.DMatrix(X_test, Y_test, enable_categorical=True)

        return dtrain_clf, dval_clf, dtest_clf
    
    else:

        return X_train, Y_train, X_val, Y_val, X_test, Y_test

## P-A

#### Train XGB

In [ ]:
for sp in tqdm(species):

   dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'pa')
   evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

   params = {"objective": "binary:logistic", "tree_method": "hist", "device": "cuda","subsample":0.3,"max_depth":5,"eval_metric":"auc","base_score":0.5}
   n=500

   model = xgb.train(
      params=params,
      dtrain=dtrain_clf,
      num_boost_round=n,
      evals=evals,
      verbose_eval=10
   )

   targets = (groundtruth[sp] > 0).astype(float)
   Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
   Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
   Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

   print(sp)
   pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'pa' / 'xgb-preds' / f"train_{sp.replace('/','-')}.csv")
   pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'pa' / 'xgb-preds' / f"val_{sp.replace('/','-')}.csv")
   pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'pa' / 'xgb-preds' / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

[0]	train-auc:0.85395	validation-auc:0.75227
[10]	train-auc:0.99197	validation-auc:0.91382
[20]	train-auc:0.99683	validation-auc:0.91057
[30]	train-auc:0.99793	validation-auc:0.91168
[40]	train-auc:0.99835	validation-auc:0.91296
[50]	train-auc:0.99863	validation-auc:0.90554
[60]	train-auc:0.99896	validation-auc:0.90131
[70]	train-auc:0.99901	validation-auc:0.89913
[80]	train-auc:0.99912	validation-auc:0.88964
[90]	train-auc:0.99922	validation-auc:0.90415
[100]	train-auc:0.99925	validation-auc:0.90711
[110]	train-auc:0.99932	validation-auc:0.89735
[120]	train-auc:0.99932	validation-auc:0.89723
[130]	train-auc:0.99939	validation-auc:0.89951
[140]	train-auc:0.99941	validation-auc:0.89199
[150]	train-auc:0.99938	validation-auc:0.89021
[160]	train-auc:0.99943	validation-auc:0.89764
[170]	train-auc:0.99944	validation-auc:0.88748
[180]	train-auc:0.99947	validation-auc:0.88798
[190]	train-auc:0.99947	validation-auc:0.88486
[200]	train-auc:0.99949	validation-auc:0.88024
[210]	train-auc:0.99954	

#### Train RF

In [54]:
for sp in tqdm(species):

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa')
    regr = RandomForestClassifier(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

    # Export predictions
    if regr.n_classes_ == 2:
        pd.Series(regr.predict_proba(X_train)[:, 1], name=sp, index = Y_train.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict_proba(X_val)[:, 1], name=sp, index = Y_val.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict_proba(X_test)[:, 1], name=sp, index = Y_test.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv")
    else:
        pd.Series([0] * len(Y_train), name=sp, index = Y_train.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"train_{sp.replace('/','-')}.csv")
        pd.Series([0] * len(Y_val), name=sp, index = Y_val.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"val_{sp.replace('/','-')}.csv")
        pd.Series([0] * len(Y_test), name=sp, index = Y_test.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

#### Binarization

In [62]:
def rvalue(thres, df, target_sr):
    sr = (df > thres).astype(int).sum(axis=1)
    
    return np.abs(sr.mean() - target_sr.mean())


xgbdict, rfdict = {}, {}

for sp in tqdm(species):

    xgbdict[sp] = pd.read_csv(root / 'pa' / 'xgb-preds' / f"val_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    rfdict[sp] = pd.read_csv(root / 'pa' / 'rf-preds' / f"val_{sp.replace('/','-')}.csv", index_col=0).squeeze()

xgb_val = pd.DataFrame(xgbdict)
rf_val = pd.DataFrame(rfdict)

targets_sr = (fulldf.loc[xgb_val.index, xgb_val.columns] > 0).sum(axis=1)
xgb_threshold = minimize_scalar(rvalue, args=(xgb_val, targets_sr),method='Bounded', bounds=(0,1))['x']
rf_threshold = minimize_scalar(rvalue, args=(rf_val, targets_sr),method='Bounded', bounds=(0,1))['x']

print(xgb_threshold, rf_threshold)

  0%|          | 0/817 [00:00<?, ?it/s]

0.057007349277399415 0.23083401501100842


#### Calculate F1 on test set

In [63]:
xgbdict, rfdict = {}, {}

for sp in tqdm(species):

    xgbdict[sp] = pd.read_csv(root / 'pa' / 'xgb-preds' / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    rfdict[sp] = pd.read_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()


xgb_test = (pd.DataFrame(xgbdict) > xgb_threshold)
rf_test = (pd.DataFrame(rfdict) > rf_threshold)


xgbdict, rfdict = {}, {}

for s in species:
    targ = (fulldf.loc[xgb_test.index, s] > 0).astype(int).to_numpy().flatten()

    xgb_preds = xgb_test[s].astype(int).to_numpy().flatten()
    xgbdict[s] = {'f1': f1_score(targ, xgb_preds, zero_division=0)}

    rf_preds = rf_test[s].astype(int).to_numpy().flatten()
    rfdict[s] = {'f1': f1_score(targ, rf_preds, zero_division=0)}



xgb_f1 = pd.DataFrame(xgbdict).T.sort_values('f1', ascending=False)
rf_f1 = pd.DataFrame(rfdict).T.sort_values('f1', ascending=False)

xgb_f1.to_csv(root / 'pa' / 'xgb_f1_scores.csv')
rf_f1.to_csv(root / 'pa' / 'rftv_f1_scores.csv')

  0%|          | 0/817 [00:00<?, ?it/s]

#### Plot comparison chart

In [64]:
xgb_f1 = pd.read_csv(root / 'pa' / 'xgb_f1_scores.csv', index_col = 0)
rf_f1 = pd.read_csv(root / 'pa' / 'rf_f1_scores.csv', index_col = 0)
rftv_f1 = pd.read_csv(root / 'pa' / 'rftv_f1_scores.csv', index_col = 0)

cp = '26_hum_env_dhw_bathy_common_pa-2025-11-10_16-31'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

cp = '37_mm4_transductif-2025-12-02_12-47'
td_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

In [65]:
scores = pd.concat([rf_f1["f1"], rftv_f1["f1"], xgb_f1["f1"], mm_f1["f1"], td_f1["f1"]], axis = 1)
names = ['RF', 'RF-TV', 'XGBoost', 'Deep-SDM', 'Transductive']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'F1']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [66]:
import plotly.express as px

fig = px.line(data[data['x'] <= 400], x='x', y='F1', color='model', template = 'simple_white',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    xaxis_title='Species rank',
    yaxis_title='F1 score',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)

fig.update_traces(line={'width': 4})

## Bins

## Reg